In [1]:
import pandas as pd 
import numpy as np 
import os 
import cv2
from psychopy import visual, core, event, gui, data, monitors


pygame 2.6.1 (SDL 2.28.4, Python 3.9.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


2026-01-08 12:59:55.744 Python[1215:13246] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/hx/_mmjyqq16qz4j0s1w8_drb5c0000gn/T/com.apple.python3.savedState
/Users/olgagulka/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## preprocessing eye tracking data:
#### - converting TIMETICK to seconds
#### - aligning eye tracking data with the start of the movie (cutting first couple of seconds of eye tracking data to include coordinates only related to the movie) and the end of the movie
#### - replacing negative numbers (blinks) with NANs and doing interpolation
#### - scaling x and y coordinates to the screen resolution

In [ ]:
#specific for each participant and movie type
folder = "guest"
participant_id = "pilot_07"  # Adjust per participant
movie_type = "guest_run_02" # Adjust per movie type
video_name = "NO_CAL_GUEST_part02.mp4" #name of the movie
start = 750493.835 # the value from which the movie start as saved in time_log_movies file I record separately 


# File paths

eye_data_file = f"/Users/olgagulka/Desktop/IBS/memory_experiment/raw_results/eye_tracking/eye_tracking_exported/{participant_id}/{folder}/{participant_id}_{movie_type}/User 0_all_gaze.csv"  #User 0_all_gaze.csv   0_fixations.csv
output_dir = "/Users/olgagulka/Desktop/IBS/memory_experiment/processed_data/eye_tracking_filtered/" 




from pymediainfo import MediaInfo
#"/Users/olgagulka/Desktop/IBS/eye_tracking/movies/SPROUT_NATPAC_02.mp4"  # for pilot 01 

video_file = MediaInfo.parse(f"/Users/olgagulka/Desktop/IBS/eye_tracking/movies_withoutCali/{video_name}")    #"/Users/olgagulka/Desktop/IBS/eye_tracking/movies_withoutCali/sprout_part_02_NoCal.mp4"
for track in video_file.tracks:
    if track.track_type == "General":
        duration_ms = track.duration
        video_duration = duration_ms / 1000
        print(f"Duration: {video_duration:.2f} seconds")



# Load and preprocess eye-tracking data
eye_data = pd.read_csv(eye_data_file)

columns = ["FPOGX", "FPOGY", "TIMETICK(f=10000000)"] #"TIME(2025/03/26 11:06:32.014)    "TIMETICK(f=10000000)"

coord_data = eye_data[columns].copy()



# Convert TIMETICK to seconds


coord_data["timestamp"] = coord_data["TIMETICK(f=10000000)"] / 10000000


# Find index of the closest value in "timestamp"
closest_idx = (coord_data["timestamp"] - start).abs().idxmin()

# Keep only rows from that index onward
coord_data = coord_data.loc[closest_idx:].copy()

# Reset "timestamp" so that the first kept row = 0
coord_data["timestamp"] = coord_data["timestamp"] - coord_data["timestamp"].iloc[0]

#reset the dataframe index to start from 0
coord_data.reset_index(drop=True, inplace=True)

print(coord_data.head())



# Trim to match video duration

coord_data = coord_data[coord_data["timestamp"] <= video_duration]


#counting how many zero or negative values in the dataset
count_negatives_x = (coord_data["FPOGX"] < 0).sum()
count_negatives_y = (coord_data["FPOGY"] < 0).sum()
count_zeros_x = (coord_data["FPOGX"] == 0).sum()
count_zeros_y = (coord_data["FPOGY"] == 0).sum()

#replacing the zero, negative or higher than 1 values with NaN (important for interpolation)
coord_data["FPOGX"] = coord_data["FPOGX"].mask((coord_data["FPOGX"] <= 0) | (coord_data["FPOGX"] > 1), np.nan)
coord_data["FPOGY"] = coord_data["FPOGY"].mask((coord_data["FPOGY"] <= 0) | (coord_data["FPOGY"] > 1), np.nan)


#interpolating the invalid Nan values with the linear function based on the values before and after each missing sample
coord_data["FPOGX"] = coord_data["FPOGX"].interpolate(method='linear', limit_direction='both')
coord_data["FPOGY"] = coord_data["FPOGY"].interpolate(method='linear', limit_direction='both')


#optoionally I can cl1ip the values to be between 0 and 1, in case the interpolation overshoots if there is a sharp jump in eye gaze
#coord_data["FPOGX"] = coord_data["FPOGX"].clip(0, 1)
#coord_data["FPOGY"] = coord_data["FPOGY"].clip(0, 1)


#screen resolution
screen_width = 1024
screen_height = 768  

# Compute pixel coordinates
coord_data["x"] = coord_data["FPOGX"] * screen_width


coord_data["y"] = (1 - coord_data["FPOGY"]) * screen_height #scaling the y's coordinates


# Apply the filter to the entire DataFrame
filtered_data = []
filtered_data = coord_data[["x", "y", "timestamp"]]

#checking if the x and y are within the screen resolution
print("x and y coordinates, resepctively maximum value (to see if is within screen resolution", filtered_data["x"].max(), filtered_data["y"].max())
print("x and y coordinates, resepctively minimum value (to see if is within screen resolution:", filtered_data["x"].min(), "," ,filtered_data["y"].min())

# checking the length of the filtered data
print(f"Trimmed data points: {len(filtered_data)}")

#double check that there are no neative numbers left
negative_count = (filtered_data["x"] < 0).sum()
negative_count_y = (filtered_data["y"] < 0).sum()
print("Number of negative values in x_coord and y_coord:", negative_count, negative_count_y)

zeros = (filtered_data["x"] == 0).sum()
zeros_y = (filtered_data["x"] == 0).sum()
print("Number of zeros in x_coord", zeros, zeros_y)



filtered_some_data = filtered_data[["timestamp", "x", "y"]].copy()


# save the filtered data to a new CSV file
saved_data_file = os.path.join(os.path.dirname(output_dir),f"{movie_type}_{participant_id}_filtered_eye_data.csv") # fixations or all gaze 
filtered_some_data.to_csv(saved_data_file, index=False)
print(f"Filtered eye-tracking data saved to {saved_data_file}")


Duration: 496.30 seconds
     FPOGX    FPOGY  TIMETICK(f=10000000)  timestamp
0  0.48318  0.38539         7504938300461   0.000000
1  0.50151  0.36538         7504938464026   0.016356
2  0.51635  0.35900         7504938629610   0.032915
3  0.52575  0.36082         7504938797315   0.049685
4  0.53411  0.35715         7504938955024   0.065456
x and y coordinates, resepctively maximum value (to see if is within screen resolution 957.71648 767.66976
x and y coordinates, resepctively minimum value (to see if is within screen resolution: 79.9744 , 0.09983999999997195
Trimmed data points: 30209
Number of negative values in x_coord and y_coord: 0 0
Number of zeros in x_coord 0 0
Filtered eye-tracking data saved to /Users/olgagulka/Desktop/IBS/memory_experiment/processed_data/eye_tracking_filtered/guest_run_02_pilot_07_filtered_eye_data.csv


: 

In [ ]:
#plotting the gaze points over the time series 
import matplotlib.pyplot as plt
import pandas as pd

x = filtered_some_data["x"]
y = filtered_some_data["y"]
time = filtered_some_data["timestamp"]
plt.figure(figsize=(12, 6))
plt.scatter(time, x, label="X coordinates", color='blue')
plt.scatter(time, y, label="Y coordinates", color='pink')
plt.xlabel("Time (s)")
plt.ylabel("Gaze Coordinates")
plt.title("Gaze Coordinates Over Time")

plt.legend(["X", "Y"])

plt.show()



,FPOGX,FPOGY,TIMETICK(f=10000000),timestamp,x,y
0,0.51640,0.37812,6569115956914,0.000000,528.79360,477.60384
1,0.58634,0.37238,6569120392848,0.443593,600.41216,482.01216
2,0.56526,0.34248,6569123678151,0.772124,578.82624,504.97536
3,0.57433,0.34568,6569128935562,1.297865,588.11392,502.51776
4,0.48144,0.30090,6569131727903,1.577099,492.99456,536.90880
...,...,...,...,...,...,...
1582,0.43116,0.30574,6575438134876,632.217796,441.50784,533.19168
1583,0.44020,0.28468,6575442081357,632.612444,450.76480,549.36576
1584,0.51987,0.31914,6575446185343,633.022843,532.34688,522.90048
1585,0.51901,0.29542,6575450126298,633.416938,531.46624,541.11744


## Merging movie parts 

Each movie has two parts that need to be combined

In [ ]:
import os
import pandas as pd

def merge_movie_parts(movie_type: str, participant_id: str):
    """
    Merge two parts (run_01 and run_02) of the same movie for a given participant.
    Adds a 'movie_part' column indicating the source (e.g., sprout_run_01 or sprout_run_02).
    
    Parameters
    ----------
    movie_type : str
        The base name of the movie (e.g., 'sprout' or 'prayer').
    participant_id : str
        The participant identifier (e.g., 'pilot_05').
    
    Saves
    -----
    A merged CSV file to:
    /Users/olgagulka/Desktop/IBS/memory_experiment/processed_data/eye_tracking_filtered/merged_parts/{movie_type}/{movie_type}_{participant_id}_merged_eye_data.csv
    """
    
    base_dir = "/Users/olgagulka/Desktop/IBS/memory_experiment/processed_data/eye_tracking_filtered"
    part_01 = f"{movie_type}_run_01"
    part_02 = f"{movie_type}_run_02"
    
    dir_01 = os.path.join(base_dir, part_01, f"{part_01}_{participant_id}_filtered_eye_data.csv")
    dir_02 = os.path.join(base_dir, part_02, f"{part_02}_{participant_id}_filtered_eye_data.csv")
    
    if not os.path.exists(dir_01):
        raise FileNotFoundError(f"Missing file: {dir_01}")
    if not os.path.exists(dir_02):
        raise FileNotFoundError(f"Missing file: {dir_02}")
    
    # Load and label data
    data_01 = pd.read_csv(dir_01)
    data_01["movie_part"] = part_01
    
    data_02 = pd.read_csv(dir_02)
    data_02["movie_part"] = part_02
    
    # Concatenate data
    merged_data = pd.concat([data_01, data_02], ignore_index=True)
    
    # Create output directory if not exists
    output_dir = os.path.join(base_dir, "merged_parts", movie_type)
    os.makedirs(output_dir, exist_ok=True)
    
    # Save merged data
    output_path = os.path.join(output_dir, f"{movie_type}_{participant_id}_merged_eye_data.csv")
    merged_data.to_csv(output_path, index=False)
    
    print(f"Merged data with 'movie_part' saved successfully at:\n{output_path}")
